In [0]:
import requests

# Retrieve API key securely from Databricks
API_KEY = dbutils.secrets.get(
    scope="football-api",
    key="football-api-key"
)

# Football API endpoint
url = "https://api.football-data.org/v4/competitions/PL"

# Authenticate with the API
response = requests.get(
    url,
    headers={"X-Auth-Token": API_KEY},
    timeout=15
)

# Check the response
print("HTTP Status:", response.status_code)

if response.status_code == 200:
    data = response.json()
    print("Competition:", data.get("name"))
    print("Country:", data.get("area", {}).get("name"))
    print("Current season:", data.get("currentSeason", {}).get("startDate"))
else:
    print("API request failed:", response.text[:300])

In [0]:
import requests
import json
import time
from datetime import datetime, timezone

# STEP 1: Retrieve API key securely
API_KEY = dbutils.secrets.get(
    scope="football-api",
    key="football-api-key"
)

# STEP 2: Define our five API endpoints
BASE_URL = "https://api.football-data.org/v4"

endpoints = {
    "competition": "/competitions/PL",
    "teams": "/competitions/PL/teams",
    "matches": "/competitions/PL/matches",
    "standings": "/competitions/PL/standings",
    "scorers": "/competitions/PL/scorers"
}

# STEP 3: Define our storage location
VOLUME_PATH = "/Volumes/workspace/default/football_raw"

# One timestamp shared by all five datasets
timestamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

# STEP 4: Extract and save each dataset
for index, (dataset_name, endpoint) in enumerate(endpoints.items()):

    print(f"Extracting {dataset_name}...")

    url = BASE_URL + endpoint

    response = requests.get(
        url,
        headers={"X-Auth-Token": API_KEY},
        timeout=30
    )

    # Stop the pipeline if the API request fails
    response.raise_for_status()

    # Convert the response into a dictionary
    data = response.json()

    # Generate the output filename
    file_path = (
        f"{VOLUME_PATH}/"
        f"{dataset_name}_{timestamp}.json"
    )

    # Save the raw JSON data
    with open(file_path, "w") as f:
        json.dump(data, f, indent=4)

    print(f"Saved: {file_path}")

    # Respect API rate limits between requests
    if index < len(endpoints) - 1:
        time.sleep(7)

print("All five datasets extracted successfully!")